# HotPotQA - CoT-SC + ReAct 결합 전략

ReAct 논문 (Yao et al., 2023) Section 4의 결합 전략을 구현한다.

| 방식 | 설명 |
|------|------|
| CoT-SC | CoT를 N번 샘플링 + 다수결 투표 |
| ReAct → CoT-SC | ReAct 실패 시 CoT-SC fallback |
| CoT-SC → ReAct | CoT-SC 신뢰도 낮으면 ReAct 실행 |

## 1. Setup

In [1]:
import os
import sys
import json
import random
import time

from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

from tools import search, lookup, wiki_state
from eval_utils import (
    load_hotpotqa, normalize_answer, f1_score, get_metrics, majority_vote
)

## 2. 데이터 및 프롬프트 로딩

In [2]:
data = load_hotpotqa("dev")
print(f"Loaded {len(data)} questions.")

with open("../prompts/prompts_naive.json", "r", encoding="utf-8") as f:
    prompt_dict = json.load(f)

# CoT 프롬프트
COT_INSTRUCTION = """질문에 대해 단계별로 추론하여 답을 제시하시오.
Thought에서 추론한 뒤 Answer에 간결한 답(단어 또는 짧은 구문)을 작성하시오.
다음은 몇 가지 예시이다.
"""
COT_PROMPT = COT_INSTRUCTION + prompt_dict["cotqa_simple6"]

# ReAct 프롬프트
REACT_INSTRUCTION = """Thought, Action, Observation 단계를 번갈아가며 수행하여 질문 응답 과제를 해결하시오.

Thought는 현재 상황에 대해 추론하는 단계이며,
Action은 다음 세 가지 유형 중 하나여야 한다:

(1) Search[entity]: Wikipedia에서 해당 entity를 검색.
(2) Lookup[keyword]: 현재 문서에서 keyword가 포함된 다음 문장을 반환.
(3) Finish[answer]: 최종 답을 반환하고 과제를 종료.

다음은 몇 가지 예시이다.
"""
REACT_PROMPT = REACT_INSTRUCTION + prompt_dict["webthink_simple6"]

print(f"CoT prompt: {len(COT_PROMPT)} chars")
print(f"ReAct prompt: {len(REACT_PROMPT)} chars")

Loaded 7405 questions.
CoT prompt: 2038 chars
ReAct prompt: 6171 chars


## 3. LLM 정의

In [3]:
# CoT-SC용: temperature > 0 (다양한 샘플링)
llm_sampling = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.7,
    max_tokens=512,
)

# ReAct용: temperature = 0 (결정적)
llm_greedy = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    max_tokens=1000,
)

print("LLMs initialized.")

LLMs initialized.


## 4. CoT-SC 구현

In [4]:
def parse_cot_answer(response_text: str) -> str:
    """CoT 응답에서 'Answer: ...' 부분을 추출한다."""
    for line in response_text.strip().split("\n"):
        line = line.strip()
        if line.lower().startswith("answer:"):
            return line.split(":", 1)[1].strip()
    # Answer: 가 없으면 마지막 줄을 답으로 사용
    lines = [l.strip() for l in response_text.strip().split("\n") if l.strip()]
    return lines[-1] if lines else ""


def cot_sc(
    question: str,
    n_samples: int = 5,
    to_print: bool = False,
) -> tuple[str, float, list[str]]:
    """CoT Self-Consistency: N번 샘플링 + 다수결 투표.

    Returns: (final_answer, confidence, all_answers)
    """
    prompt = COT_PROMPT + f"Question: {question}\n"
    answers = []

    for i in range(n_samples):
        response = llm_sampling.invoke(prompt, stop=["\n\n"])
        answer = parse_cot_answer(response.content)
        answers.append(answer)
        if to_print:
            print(f"  [Sample {i+1}] {response.content.strip()[:150]}")
            print(f"  → Answer: {answer}")

    final_answer, confidence = majority_vote(answers)

    if to_print:
        print(f"\n  Votes: {answers}")
        print(f"  => Final: {final_answer} (confidence: {confidence:.0%})")

    return final_answer, confidence, answers


print("CoT-SC defined.")

CoT-SC defined.


## 5. ReAct 구현 (LangGraph)

In [5]:
class AgentState(TypedDict):
    question: str
    prompt: str
    current_step: int
    max_steps: int
    answer: str
    done: bool
    trajectory: str
    n_calls: int
    n_badcalls: int


def reasoning_node(state: AgentState) -> dict:
    step = state["current_step"]
    prompt = state["prompt"]
    query = prompt + f"Thought {step}:"
    response = llm_greedy.invoke(query, stop=[f"\nObservation {step}:"])
    thought_action = response.content.strip()

    n_calls = state["n_calls"] + 1
    n_badcalls = state["n_badcalls"]

    try:
        thought, action = thought_action.split(f"\nAction {step}: ")
    except ValueError:
        n_badcalls += 1
        n_calls += 1
        thought = thought_action.split("\n")[0]
        action_response = llm_greedy.invoke(
            prompt + f"Thought {step}: {thought}\nAction {step}:",
            stop=["\n"],
        )
        action = action_response.content.strip()

    if action:
        action = action[0].lower() + action[1:]

    return {
        "prompt": prompt + f"Thought {step}: {thought}\nAction {step}: {action}\n",
        "trajectory": state["trajectory"] + f"Thought {step}: {thought}\nAction {step}: {action}\n",
        "n_calls": n_calls,
        "n_badcalls": n_badcalls,
    }


def tool_exec_node(state: AgentState) -> dict:
    step = state["current_step"]
    trajectory = state["trajectory"]

    lines = trajectory.strip().split("\n")
    action_line = ""
    for line in reversed(lines):
        if line.startswith(f"Action {step}:"):
            action_line = line.split(": ", 1)[1] if ": " in line else ""
            break
    action = action_line.strip()

    if action.startswith("search[") and action.endswith("]"):
        entity = action[len("search["):-1]
        obs = search.invoke({"entity": entity})
    elif action.startswith("lookup[") and action.endswith("]"):
        keyword = action[len("lookup["):-1]
        obs = lookup.invoke({"keyword": keyword})
    elif action.startswith("finish[") and action.endswith("]"):
        answer = action[len("finish["):-1]
        obs_text = f"Observation {step}: Episode finished.\n"
        return {
            "answer": answer, "done": True,
            "prompt": state["prompt"] + obs_text,
            "trajectory": trajectory + obs_text,
            "current_step": step + 1,
        }
    else:
        obs = f"Invalid action: {action}"

    obs = obs.replace("\\n", "")
    obs_text = f"Observation {step}: {obs}\n"
    return {
        "prompt": state["prompt"] + obs_text,
        "trajectory": trajectory + obs_text,
        "current_step": step + 1,
    }


def should_continue(state: AgentState) -> str:
    if state.get("done", False):
        return "end"
    if state["current_step"] > state["max_steps"]:
        return "end"
    return "continue"


graph = StateGraph(AgentState)
graph.add_node("reasoning", reasoning_node)
graph.add_node("tool_exec", tool_exec_node)
graph.add_edge(START, "reasoning")
graph.add_edge("reasoning", "tool_exec")
graph.add_conditional_edges("tool_exec", should_continue, {"continue": "reasoning", "end": END})
react_app = graph.compile()

print("ReAct graph compiled.")

ReAct graph compiled.


In [6]:
def react(question: str, to_print: bool = False, max_steps: int = 7) -> tuple[str, dict]:
    """ReAct 에이전트를 실행한다."""
    wiki_state.reset()
    initial_state = {
        "question": question,
        "prompt": REACT_PROMPT + f"Question: {question}\n",
        "current_step": 1,
        "max_steps": max_steps,
        "answer": "",
        "done": False,
        "trajectory": f"Question: {question}\n",
        "n_calls": 0,
        "n_badcalls": 0,
    }
    result = react_app.invoke(initial_state)

    answer = result["answer"]
    info = {
        "answer": answer,
        "steps": result["current_step"] - 1,
        "n_calls": result["n_calls"],
        "n_badcalls": result["n_badcalls"],
        "done": result["done"],
        "trajectory": result["trajectory"],
    }

    if to_print:
        print(f"Q: {question}")
        print(result["trajectory"])
        print(f"=> Answer: {answer}")

    return answer, info


print("ReAct function defined.")

ReAct function defined.


## 6. 결합 전략 구현

In [7]:
def react_then_cot_sc(
    question: str,
    n_samples: int = 5,
    to_print: bool = False,
) -> tuple[str, dict]:
    """방식 A: ReAct 먼저 실행 → 실패 시 CoT-SC fallback."""
    answer, info = react(question, to_print=to_print)

    # ReAct 실패 판정
    react_failed = (
        not info["done"]                              # finish 안 함
        or not answer                                  # 답 없음
        or "Invalid action" in info["trajectory"]      # 잘못된 액션
    )

    if react_failed:
        if to_print:
            print(f"  [ReAct failed] Falling back to CoT-SC...")
        cot_answer, confidence, all_answers = cot_sc(
            question, n_samples=n_samples, to_print=to_print
        )
        info["method"] = "cot_sc_fallback"
        info["cot_confidence"] = confidence
        info["cot_answers"] = all_answers
        return cot_answer, info
    else:
        info["method"] = "react"
        return answer, info


def cot_sc_then_react(
    question: str,
    n_samples: int = 5,
    confidence_threshold: float = 0.5,
    to_print: bool = False,
) -> tuple[str, dict]:
    """방식 B: CoT-SC 먼저 실행 → 신뢰도 낮으면 ReAct 실행."""
    cot_answer, confidence, all_answers = cot_sc(
        question, n_samples=n_samples, to_print=to_print
    )

    info = {
        "cot_answer": cot_answer,
        "cot_confidence": confidence,
        "cot_answers": all_answers,
    }

    if confidence >= confidence_threshold:
        if to_print:
            print(f"  [CoT-SC confident ({confidence:.0%})] Using CoT-SC answer.")
        info["method"] = "cot_sc"
        info["answer"] = cot_answer
        return cot_answer, info
    else:
        if to_print:
            print(f"  [CoT-SC low confidence ({confidence:.0%})] Running ReAct...")
        react_answer, react_info = react(question, to_print=to_print)
        info.update(react_info)
        info["method"] = "react_after_low_confidence"
        return react_answer, info


print("Combination strategies defined.")

Combination strategies defined.


## 7. 단일 질문 테스트 (4가지 방식 비교)

In [8]:
question, gt_answer = data[0]
print(f"Question: {question}")
print(f"GT: {gt_answer}\n")
print("=" * 60)

# 1) ReAct only
print("\n[1] ReAct only")
pred1, info1 = react(question, to_print=True)
print(f"EM: {normalize_answer(pred1) == normalize_answer(gt_answer)}")

print("\n" + "=" * 60)

# 2) CoT-SC only
print("\n[2] CoT-SC only (n=5)")
pred2, conf2, _ = cot_sc(question, n_samples=5, to_print=True)
print(f"EM: {normalize_answer(pred2) == normalize_answer(gt_answer)}")

print("\n" + "=" * 60)

# 3) ReAct → CoT-SC
print("\n[3] ReAct → CoT-SC fallback")
pred3, info3 = react_then_cot_sc(question, to_print=True)
print(f"Method: {info3['method']}")
print(f"EM: {normalize_answer(pred3) == normalize_answer(gt_answer)}")

print("\n" + "=" * 60)

# 4) CoT-SC → ReAct
print("\n[4] CoT-SC → ReAct (threshold=0.5)")
pred4, info4 = cot_sc_then_react(question, confidence_threshold=0.5, to_print=True)
print(f"Method: {info4['method']}")
print(f"EM: {normalize_answer(pred4) == normalize_answer(gt_answer)}")

Question: Were Scott Derrickson and Ed Wood of the same nationality?
GT: yes


[1] ReAct only
Q: Were Scott Derrickson and Ed Wood of the same nationality?
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Thought 1: Thought 1: I need to search Scott Derrickson and Ed Wood, find their nationalities, then determine if they are the same.

Action 1: search[Scott Derrickson]
Observation 1: Scott Derrickson (born July 16, 1966) is an American filmmaker. He is known for his work in the horror genre, directing films such as The Exorcism of Emily Rose (2005), Sinister (2012), The Black Phone (2021), and its sequel, Black Phone 2 (2025). He is also known for the superhero film Doctor Strange (2016), based on the Marvel Comics character.. Scott Derrickson grew up in Denver, Colorado. He graduated from Biola University with a BA in Humanities with an emphasis in philosophy and literature and a B.A.
Thought 2: Thought 2: Scott Derrickson is American. I need to search Ed Wood and

## 8. 배치 비교 평가 (10개 샘플)

4가지 방식을 동일 샘플에 대해 평가한다.  
CoT-SC가 포함된 방식은 API 호출이 많으므로, 소수로 먼저 테스트를 권장한다.

In [9]:
N_EVAL = 10  # 평가할 샘플 수 (소수로 테스트 시 10~20으로 변경)
N_SC_SAMPLES = 5  # CoT-SC 샘플링 횟수
CONFIDENCE_THRESHOLD = 0.5

idxs = list(range(len(data)))
random.Random(233).shuffle(idxs)

methods = {
    "ReAct": lambda q: react(q),
    "CoT-SC": lambda q: (lambda r: (r[0], {"answer": r[0], "confidence": r[1]}))(cot_sc(q, N_SC_SAMPLES)),
    "ReAct→CoT-SC": lambda q: react_then_cot_sc(q, N_SC_SAMPLES),
    "CoT-SC→ReAct": lambda q: cot_sc_then_react(q, N_SC_SAMPLES, CONFIDENCE_THRESHOLD),
}

all_results = {name: [] for name in methods}
old_time = time.time()

for i, idx in enumerate(idxs[:N_EVAL]):
    question, gt_answer = data[idx]
    print(f"\n[{i+1}/{N_EVAL}] Q: {question}")
    print(f"  GT: {gt_answer}")

    for method_name, method_fn in methods.items():
        try:
            pred, info = method_fn(question)
        except Exception as e:
            print(f"  {method_name}: ERROR - {e}")
            pred = ""

        metrics = get_metrics(pred, gt_answer)
        all_results[method_name].append(metrics)

        em_sum = sum(r["em"] for r in all_results[method_name])
        avg_em = em_sum / len(all_results[method_name])
        print(f"  {method_name:15s}: {pred[:50]:50s} | EM={avg_em:.3f} ({em_sum}/{len(all_results[method_name])})")

    elapsed = (time.time() - old_time) / (i + 1)
    print(f"  ({elapsed:.1f}s/q)")
    print("-" * 80)
    sys.stdout.flush()


[1/10] Q: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
  GT: Beyond the Clouds
  ReAct          :                                                    | EM=0.000 (0/1)
  CoT-SC         : Thought: Let's think step by step. The American ac | EM=0.000 (0/1)
  ReAct→CoT-SC   : Thought: Let's think step by step. The American ac | EM=0.000 (0/1)
  CoT-SC→ReAct   :                                                    | EM=0.000 (0/1)
  (83.9s/q)
--------------------------------------------------------------------------------

[2/10] Q: Who created the show with Wendy Schaal doing the voice of Francine?
  GT: Seth MacFarlane
  ReAct          : Seth MacFarlane, Mike Barker, and Matt Weitzman    | EM=0.000 (0/2)
  CoT-SC         : Thought: Let's think step by step. Wendy Schaal vo | EM=0.000 (0/2)
  ReAct→CoT-SC   : Seth MacFarlane, Mike Barker, and Matt Weitzman    | EM=0.000 (0/2)
  CoT-SC→ReAct   : Seth MacFarlane, Mike Barke

KeyboardInterrupt: 

## 9. 최종 결과 비교

In [ ]:
print(f"\n{'='*60}")
print(f"  Final Results ({N_EVAL} samples, CoT-SC n={N_SC_SAMPLES})")
print(f"{'='*60}")
print(f"{'Method':20s} {'EM':>8s} {'F1':>8s}")
print(f"{'-'*20} {'-'*8} {'-'*8}")

for method_name, results in all_results.items():
    n = len(results)
    if n == 0:
        continue
    total_em = sum(r["em"] for r in results)
    total_f1 = sum(r["f1"] for r in results)
    print(f"{method_name:20s} {total_em/n:8.3f} {total_f1/n:8.3f}")

print(f"{'='*60}")

## 10. 결과 저장

In [ ]:
os.makedirs("trajs", exist_ok=True)

summary = {}
for method_name, results in all_results.items():
    n = len(results)
    if n == 0:
        continue
    summary[method_name] = {
        "n_samples": n,
        "em": sum(r["em"] for r in results) / n,
        "f1": sum(r["f1"] for r in results) / n,
    }

with open("trajs/cot_sc_comparison.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved to trajs/cot_sc_comparison.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))